### Regresión logística binaria (*benchmark*)

Se planteará inicialmente un modelo base de sencilla aplicación: **modelo de regresión logística binario**, teniendo en cuenta solo los efectos principales de cada variable. Posteriormente se comparará dicho modelo con otras opciones más complejas basadas en *boosting*

#### Librerías


In [17]:
import polars as pl
from pathlib import Path
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegressionCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

#### Carga de datos

In [8]:
ROOT = Path('..')
datos_limpios = pl.read_parquet(ROOT/"data"/"processed"/"temporada1_limpio.parquet")
datos_limpios_t2 = pl.read_parquet(ROOT/"data"/"processed"/"temporada2_limpio.parquet")

#### Preselección de variables

Como variables predictoras se utilizarán aquellas definidas durante el proceso de feature engineering, excluyendo los identificadores (`pitch_id`), los identificadores de los bateadores (`batter`) y lanzadores (`pitcher`), las variables `plate_x` y `plate_z` (ya que la información que contienen queda representada por `relative_x` y `relative_height`, respectivamente) y la variable distance_to_corner, debido a que aproximadamente el 60% de sus observaciones toman el valor cero, al corresponder a lanzamientos realizados fuera de la zona de strike.

De esta forma, el modelo se ajusta utilizando 22 variables explicativas, de las cuales 16 son numéricas, 3 binarias y 3 categóricas.

In [9]:
numeric_features = [
    "release_speed",
    "balls",
    "strikes",
    "pfx_x",
    "pfx_z",
    "plate_x",
    "plate_z",
    "sz_top",
    "sz_bot",
    "sz_mid",
    "strike_zone_size",
    "movement_complexity",
    "relative_height",
    "relative_x",
    "pitch_location",
    "complex_speed",
]

binary_features = [
    "platoon_advantage",
    "is_strike_zone",
    "is_shadow_zone",
]

categorical_features = [
    "pitch_type",
    "stand",
    "p_throws",
]

X = datos_limpios.select(
    numeric_features +
    binary_features +
    categorical_features
)

y = datos_limpios["swing"]

#### Matriz de diseño

Para armar la matriz de diseño utilizada en el modelo se realizó previamente un proceso de preprocesamiento de los datos. Al observarse datos faltantes tanto en las variables numéricas como en las categóricas se decide realizar imputación por la **mediana** y la **moda** respectivamente. Luego de este paso, vamos a transformar nuestras variables de modo que las numéricas se *estandarizan* y las categoricas se convierten en variables *Dummy's*. Este procedimiento secuencial se implemento mediante la función `Pipeline` la cual nos garantiza que se realicen dichos procedimientos tanto en las etapas de validación como en las predicciones posteriores.

Para realizar estas tranformaciones en las columnas y obtener la matriz de diseño utilizada en el modelo, se utiliza la función `ColumnTransformer`. Mediante la cual se aplican los tratamientos mencionados a cada variable según su tipo.

In [10]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

binary_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features,
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features,
        ),
        (
            "bin",
            binary_transformer,
            binary_features,
        ),
    ]
)

#### Planteo del modelo

Se modela la probabilidad de realizar swing mediante el siguiente modelo de regresión logística:

$$
\log\left(\frac{\pi_i}{1-\pi_i}\right)
=
\beta_0+\beta_1x_{i1}+\cdots+\beta_px_{ip},
$$

donde,
 $\beta_0$ representa el intercepto,
 $\beta_1,\ldots,\beta_p$ los coeficientes asociados a las variables explicativas y
 $\pi_i$ representa la probabilidad de que el bateador en el lanzamiento $i$ realice un swing.


La estimación de los parámetros no se realizó mediante máxima verosimilitud clásica, sino incorporando una penalización de tipo cuadrática. Este enfoque reduce la magnitud de los coeficientes, disminuyendo la varianza de las estimaciones y mejorando la capacidad de generalización del modelo frente a nuevos datos (reduce el sobreajuste). Además, resulta útil cuando existen relaciones de multicolinealidad entre los predictores.

La penalización elegida fue $l2$ o como la conocemos los amigos: $Ridge$, incorporando una penalización cuadrática sobre los coeficientes del modelo. De esta manera la log-versimilitud penalizada a maximizar resulta:

$$
\hat{\boldsymbol{\beta}}
=
\argmax_{\boldsymbol{\beta}}
\left\{

\sum_{i=1}^{n}
\left[
y_i \log(\pi_i) + (1 - y_i)\log(1 - \pi_i)
\right]
-
\lambda \sum_{j=1}^{p} \beta_j^2
\right\}.
$$


In [11]:
model = LogisticRegressionCV(
    penalty="l2",
    solver="lbfgs",
    scoring="neg_log_loss",
    cv=5,
    Cs=np.logspace(-3, 3, 10),
    max_iter=1000,
    n_jobs=-1
)

El hiperparámetro de regularización se seleccionó mediante validación cruzada con cinco particiones (*5-fold cross validation*). La implementación de `LogisticRegressionCV` busca el valor óptimo de C=1/λ utilizando como criterio de selección el menor valor promedio de LogLoss obtenido sobre los conjuntos de validación. Y el algoritmo elegido para obtener la maximización fue *Limited-memory Broyden–Fletcher–Goldfarb–Shanno.*, aunque en clases solíamos utilizar *Newton-Raphson* el método utilizado en este caso es más eficiente computacionalmente (Huang, 2018).

#### Pipeline

Armo un Pipeline para trabajar de manera organizada, mediante el mismo voy a aplicar las transformaciones mencionadas anteriormente y puedo definirle que modelo quiero utilizar. En esta primera instancia el modelo de regresión logística con Ridge, con los efectos principales de las variables.

In [12]:
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model),
    ]
)

#### Entrenamiento del modelo

Con el método `.fit` entreno el pipeline completo. aA partir del cual se ajustan las transformaciones de preprocesamiento y se entrena el modelo de regresión logística, aplicando la estrategia de validación cruzada para obtener los hiperparámetros. 

Guardando los resultados de las validaciones, valores de los coeficientes, etc.

In [18]:
pipeline.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int8](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](22,)","['release_speed','balls','strikes',...,'pitch_type','stand','p_throws']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,22
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, 

Podemos observar que de las 22 variables explicativas pertenecientes a nuesta base de datos, el modelo ajusta 39 coeficientes. Dado que las 3 variables categóricas se transformaron en 20 variables Dummys.

#### Resultados

##### $\lambda$ óptimo a partir de la métrica LogLoss

Mediante el procedimiento de validación cruzada, se calculo para cads valor del hiperparámetro, sud respectiva métrica de validación en cada uno de los 5 folds. Los resultados obtenidos se muestran a continuación.

**Nota**: Scikit-Learn busca maximizar sus hiperparámetros, por lo que la función devuelve *C*  ($1/\lambda$) en vez de $\lambda$

In [14]:
model = pipeline.named_steps["model"]

scores = model.scores_[1]  # clase positiva (swing)

results = pl.DataFrame({
    "C": model.Cs_,
    "lambda": 1 / model.Cs_,
    "LogLoss": -scores.mean(axis=0)
})

results

C,lambda,LogLoss
f64,f64,f64
0.001,1000.0,0.453003
0.004642,215.443469,0.452789
0.021544,46.415888,0.452716
0.1,10.0,0.452713
0.464159,2.154435,0.452716
2.154435,0.464159,0.452716
10.0,0.1,0.452716
46.415888,0.021544,0.452716
215.443469,0.004642,0.452716


Se observa que el LogLoss resulta muy similar para los distintos $\lambda_s$ planteados. Esto nos da un indicio de que la regularización no es muy útil en este caso.

##### Coeficientes

Observamos la importancia de las variables a través de sus coeficientes estimados.

In [15]:
feature_names = pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()

coef = pipeline.named_steps["model"].coef_[0]

coef_df = pl.DataFrame({
    "Variable": feature_names,
    "Coeficiente": coef
})

coef_df_sorted = (
    coef_df
    .with_columns(
        pl.col("Coeficiente").abs().alias("abs_coef")
    )
    .sort("abs_coef", descending=True)
    .drop("abs_coef")
)

coef_df_sorted.head(10)

Variable,Coeficiente
str,f64
"""num__pitch_location""",-1.411933
"""num__strike_zone_size""",1.154233
"""num__strikes""",0.776309
"""num__sz_bot""",-0.674039
"""num__relative_height""",-0.562342
"""num__plate_z""",0.420469
"""cat__pitch_type_FS""",0.37642
"""bin__is_shadow_zone""",0.371515
"""cat__pitch_type_CH""",0.326634


Al observar a los 10 coeficientes con mayor valor absoluto podemos identificar a la variable `pitch_location` como la más influyente a la hora de predecir la probabilidad de realizar un swing. Variable generada a través del proceso de ingeniería de datos, que nos da un indicio de que, manteniendo las demás variables constantes, a mayor distancia del centro de la zona de strike, menos es la probabilidad de que el bateador realice un swing.

Por otra parte vemos que la altura de la zona de strike (`strike_zone_size`) influye positivamente en la intención de realizar un swing, siendo concordante con el análisis descriptivo observamos que a mayor tamaño de zona, mayor probabilidad de swing. No obstante, dado que la altura de la zona depende de la altura del bateador y de la delimitación realizada por el sistema de medición, este efecto debe interpretarse con cautela y no necesariamente implica una relación causal.

Por último el número de strikes es la tercer variable que más explica a la realización de un swing, observandose que a mayor cantidad de strikes previos, manteniendose constantes las demas variables, mayor probabilidad de realizar un swing. Este comportamiento coincide con el análisis descriptivo realizado anteriormente y resulta consistente con la lógica del juego, donde a medida que el bateador disponga de menor margen para dejar pasar lanzamientos, más va a batear.

**Nota**: Los cambios no reflejan un cambio directo en la probabilidad, ya lo que se modela es el logit de la probabilidad. Igualmente el efecto positivo de una variable se asocia a un cambio positivo en la probabilidad de swing.

#### Predicciones

Una vez entrenado el modelo de regresión logística con Ridge, vamos a realizar predicciones para la segunda temporada en estudio. A este dataset se le aplicaran los mismos procedimientos que al de entrenamiento y posteriormente se calcularan las probabilidades predichas.

In [16]:
pitch_ids = datos_limpios_t2["pitch_id"]

X_t2 = datos_limpios_t2.select(
    numeric_features +
    binary_features +
    categorical_features
)

swing_probability = pipeline.predict_proba(X_t2)[:, 1]

validacion = pl.DataFrame({
    "pitch_id": pitch_ids,
    "swing_probability": swing_probability,
})

validacion.write_parquet(
    ROOT / "outputs" / "regresion_logistica" / "validacion.parquet"
)

Al subir dichas predicciones a Kaggle, obtenemos un valor de Log Loss = **0.627**. 

Observamos cierta diferencia con el valor obtenido en la validación, el cual fue **0.452** lo cual nos da un indicio de que el modelo aprendió patrones de la temporada 1 no se repiten en la temporada 2.

A su vez, este modelo base tiene limitaciones y puede no alcanzar para explicar de manera óptima a nuestra variable respuesta. Aunque sigue siendo un modelo válido como punto de partida.

#### Conclusiones breves

##### Ventajas

- Sencilla aplicación
- Fácil interpretación
- "Barato" computacionalmente (1min)

##### Desventajas

- Supone linealidad en los efectos
- Supone normalidad y homocedasticidad en los errores
- Solo se plantean los efectos individuales de las variables (al tener tantas filas y variables, no se puede analizar la interacción entre las mismas)

#### Referencias

Huang, B. (2018). Gradient Descent, Newton's Method, and LBFGS. Optimization in Machine Learning Blog. https://wordpress.cs.vt.edu/optml/2018/02/03/gradient-descent-newtons-method-and-lbfgs/